# BRONZE LAYER SETUP 

In [0]:
import logging
from pyspark.sql.functions import explode, col
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("BronzeLayer")
logger.info("Starting Bronze Ingestion")

INFO:BronzeLayer:Starting Bronze Ingestion


In [0]:
BRONZE_PATH = "abfss://ecom@ecomstore.dfs.core.windows.net/bronze/"
INCREMENTAL_PATH = "abfss://ecom@ecomstore.dfs.core.windows.net/incremental/"

In [0]:
spark.sql("USE CATALOG ecom_catalog")
spark.sql("USE bronze")

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


DataFrame[]

# TRANSFORMING FUNCTION FOR CARTS

In [0]:
def transform_cart(df):
    df_cart = df.select(explode("carts").alias("cart"))
    return df_cart.select(
        col("cart.id").alias("cart_id"),
        col("cart.userId").alias("user_id"),
        explode("cart.products").alias("product")
    ).select(
        "cart_id",
        "user_id",
        col("product.id").alias("product_id"),
        col("product.title").alias("product_title"),
        col("product.quantity"),
        col("product.price"),
        col("product.total"),
        col("product.discountPercentage").alias("discount_percentage"),
        col("product.discountedTotal").alias("discounted_total")
    )

# TRANSFORM PRODUCT JSON

In [0]:
def transform_products(df):
    df_products = df.select(explode("products").alias("product"))
    return df_products.select(
        col("product.id").alias("product_id"),
        col("product.title"),
        col("product.description"),
        col("product.brand"),
        col("product.category"),
        col("product.price"),
        col("product.stock"),
        col("product.rating")
    )

# CEHCKING EXISTANCE OF TABLES 

In [0]:
cart_exists = spark.catalog.tableExists("ecom_catalog.bronze.cart_bronze")
product_exists = spark.catalog.tableExists("ecom_catalog.bronze.products_bronze")
logger.info(f"Cart table exists: {cart_exists}")
logger.info(f"Product table exists: {product_exists}")

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:BronzeLayer:Cart table exists: True
INFO:BronzeLayer:Product table exists: True
INFO:py4j.clientserver:Received command c on object id p0


# PROCESS OF FULL LOAD INCREMENTAL LOAD 

In [0]:
if not cart_exists:
    try:
        logger.info("running FULL LOAD ")
        # CART
        df_cart = spark.read.option("multiline", "true").json(BRONZE_PATH + "cart.json")
        df_cart_final = transform_cart(df_cart) # DF TRANSFROMED
        df_cart_final.write.format("delta").mode("overwrite").saveAsTable("ecom_catalog.bronze.cart_bronze")  # WRITING DOWN TABLE IN DELTA
        # PRODUCTS
        df_products = spark.read.option("multiline", "true").json(BRONZE_PATH + "products.json")
        df_products_final = transform_products(df_products)
        df_products_final.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable("ecom_catalog.bronze.products_bronze")

        logger.info("full load completed successfully.")

    except Exception as e:
        logger.error(f"error in full load: {e}")
        raise
else:
    try:
        logger.info("running INCREMENTAL LOAD.")    # starting fro incremental load

        all_files = [f.path for f in dbutils.fs.ls(INCREMENTAL_PATH) if f.name.endswith(".json")]
        if not all_files:
            logger.info("no incremental data found.")
        else:
            #  PROCESS CARTS FOR INCREMENTAL LOAD
            cart_files = [f for f in all_files if "cart" in f.lower()]
            if cart_files:
                df_cart_inc = spark.read.option("multiline", "true").json(cart_files)
                df_cart_final = transform_cart(df_cart_inc)
                df_cart_final = df_cart_final.dropDuplicates(["cart_id", "product_id"])
                df_cart_final.createOrReplaceTempView("cart_updates")
                # Use UPSERT OPERATIOSN
                spark.sql("""
                MERGE INTO ecom_catalog.bronze.cart_bronze t
                USING cart_updates s
                ON t.cart_id = s.cart_id AND t.product_id = s.product_id
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
                """)
                logger.info("Successfully merged incremental carts.")
            
           # 2. PROCESS PRODUCTS FOR INCREMENTAL LOAD (APPEND-ONLY 
            product_files = [f for f in all_files if "product" in f.lower()]
            if product_files:
                df_prod_inc = spark.read.option("multiline", "true").json(product_files)
                df_products_final = transform_products(df_prod_inc)
                # Append directly to the Bronze table
                df_products_final.write.format("delta") \
                .mode("append").saveAsTable("ecom_catalog.bronze.products_bronze")
                logger.info("Successfully appended incremental products to Bronze.")
            # 3.DELETE FILES FROM INCREMENTAL DIRECTORY AFTER UPSERT
            for file_path in all_files:
                dbutils.fs.rm(file_path)
                logger.info(f"Cleaned up processed file: {file_path}")   
            logger.info("Incremental load and cleanup completed")
    except Exception as e:
        logger.error(f"Error in incremental load: {e}")
        raise

INFO:BronzeLayer:running INCREMENTAL LOAD.
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Python Server ready to receive messages
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:R

In [0]:
%sql
select * from cart_bronze;


cart_id,user_id,product_id,product_title,quantity,price,total,discount_percentage,discounted_total
1001,404,9999,Data Engineer Coffee Mug,2,15.99,31.98,0.0,31.98
1,33,78,Apple MacBook Pro 14 Inch Space Grey,2,1999.99,3999.98,18.52,3259.18
1,33,100,Apple Airpods,5,129.99,649.95,12.84,566.5
1,33,168,Charger SXT RWD,3,32999.99,98999.97,13.39,85743.87
1,33,183,Green Oval Earring,5,24.99,124.94999999999999,6.28,117.1
2,142,110,Selfie Lamp with iPhone,5,14.99,74.95,19.87,60.06
2,142,122,iPhone 6,3,299.99,899.97,19.64,723.22
2,142,124,iPhone X,4,899.99,3599.96,8.03,3310.88
2,142,144,Cricket Helmet,4,44.99,179.96,11.47,159.32
2,142,148,Golf Ball,4,9.99,39.96,11.24,35.47


In [0]:
%sql
select * from products_bronze

product_id,title,description,brand,category,price,stock,rating
1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,Essence,beauty,9.99,99,2.56
2,Eyeshadow Palette with Mirror,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.",Glamour Beauty,beauty,19.99,34,2.86
3,Powder Canister,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.",Velvet Touch,beauty,14.99,89,4.64
4,Red Lipstick,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.",Chic Cosmetics,beauty,12.99,91,4.36
5,Red Nail Polish,"The Red Nail Polish offers a rich and glossy red hue for vibrant and polished nails. With a quick-drying formula, it provides a salon-quality finish at home.",Nail Couture,beauty,8.99,79,4.32
6,Calvin Klein CK One,"CK One by Calvin Klein is a classic unisex fragrance, known for its fresh and clean scent. It's a versatile fragrance suitable for everyday wear.",Calvin Klein,fragrances,49.99,29,4.37
7,Chanel Coco Noir Eau De,"Coco Noir by Chanel is an elegant and mysterious fragrance, featuring notes of grapefruit, rose, and sandalwood. Perfect for evening occasions.",Chanel,fragrances,129.99,58,4.26
8,Dior J'adore,"J'adore by Dior is a luxurious and floral fragrance, known for its blend of ylang-ylang, rose, and jasmine. It embodies femininity and sophistication.",Dior,fragrances,89.99,98,3.8
9,Dolce Shine Eau de,"Dolce Shine by Dolce & Gabbana is a vibrant and fruity fragrance, featuring notes of mango, jasmine, and blonde woods. It's a joyful and youthful scent.",Dolce & Gabbana,fragrances,69.99,4,3.96
10,Gucci Bloom Eau de,"Gucci Bloom by Gucci is a floral and captivating fragrance, with notes of tuberose, jasmine, and Rangoon creeper. It's a modern and romantic scent.",Gucci,fragrances,79.99,91,2.74
